# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, following the Croissant standard. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # DatasetMetadata object
print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

> **Note:** The Croissant metadata must be inspected to discover record set and field `@id` values before record loading. Let's enumerate all record sets and their fields.

In [ ]:
# List all available record sets and their fields by @id

record_sets = dataset.record_sets  # List of RecordSet objects or empty if no record sets
if not record_sets:
    print("No record sets found in this dataset. Please verify the Croissant schema or use metadata.distribution to inspect data sources.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    {field.id} ({field.data_type})")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All extraction is based on entity `@id` values for clarity and reproducibility.

In [ ]:
# Automatically extract all available record sets (if any) and load into DataFrames
import warnings
warnings.filterwarnings('ignore')

dataframes = {}
if not record_sets:
    print("No record sets available for extraction.")
else:
    record_set_ids = [rs.id for rs in record_sets]
    for record_set_id in record_set_ids:
        print(f"Loading records from Record Set @id: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"    Columns: {list(df.columns)}")
            else:
                print("    No records found in this record set.")
        except Exception as e:
            print(f"    Error loading this record set: {e}")
    if dataframes:
        # Choose first available DataFrame for further demo
        first_rs_id = next(iter(dataframes))
        print(f"\nPreview for {first_rs_id}: \n", dataframes[first_rs_id].head())
    else:
        print("No records were loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields must be referenced by their `@id`.

> **Note:** For illustration, we select a numeric field and a grouping field by their `@id` from the extracted DataFrame. Make sure to adapt the field `@id` names as seen in the DataFrame.

In [ ]:
# Example EDA if data loaded
if dataframes:
    df = next(iter(dataframes.values()))
    record_set_id = next(iter(dataframes.keys()))
    print(f"Exploring record set: {record_set_id}")
    print(f"Available columns (@id): {list(df.columns)}\n")

    # Attempting to pick typical numeric and group fields by heuristics
    numeric_field_candidates = [col for col in df.columns if (df[col].dtype in [float, int]) or ('log_likelihood' in col.lower()) or ('coefficient' in col.lower())]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field (@id): {numeric_field_id}")
    else:
        print("No numeric field detected - please review columns and update the field @id.")
        numeric_field_id = None

    group_field_candidates = [col for col in df.columns if "ward" in col.lower() or "group" in col.lower() or "gender" in col.lower()]
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    if group_field_id:
        print(f"Using grouping field (@id): {group_field_id}")

    # Filtering, normalization, and grouping
    if numeric_field_id:
        try:
            df_numeric = df[numeric_field_id].apply(pd.to_numeric, errors='coerce')
            threshold = df_numeric.mean()
            filtered_df = df[df_numeric > threshold].copy()
            print(f"Filtered records where {numeric_field_id} > {threshold:.3f} (mean):\n", filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field_id}_normalized"] = (df_numeric - df_numeric.mean()) / df_numeric.std()
            print(f"\nNormalized {numeric_field_id}:\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Grouping
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
                print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:\n", grouped_df)
        except Exception as e:
            print(f"EDA failed: {e}")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All axes must use field `@id`.

> Note: Visualization is performed only if relevant data is loaded.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    df_numeric = df[numeric_field_id].apply(pd.to_numeric, errors='coerce')
    sns.histplot(df_numeric.dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()
    
    # Boxplot or group visualization, if a grouping field exists
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* We loaded and validated the FAIR^2 dataset Croissant schema and its metadata.
* All data exploration steps referenced record set and field entities by their `@id` per Croissant standard best practices.
* Fields and data are extracted dynamically, enabling reproducible and robust downstream analyses with rich, interpretable field and record identifiers.
* For further analysis, always inspect field and record set `@id` values to ensure correct references in future workflows.